In [1]:
import torch, torchaudio, numpy as np
from pathlib import Path
from tqdm import tqdm
from speechbrain.inference import EncoderClassifier

/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [2]:
DATA_ROOT = Path("../data/librispeech/LibriSpeech/test-clean")
OUT_DIR   = Path("../outputs/librispeech_ecapa")
OUT_DIR.mkdir(parents=True, exist_ok=True)

flac_files = sorted(DATA_ROOT.rglob("*.flac"))
print("Total flac files:", len(flac_files))

Total flac files: 2620


In [3]:
# ECAPA extractor — same model as VoxCeleb experiments
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}
)

/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


In [4]:
X, utterances, speakers = [], [], []

for flac_path in tqdm(flac_files):
    signal, sr = torchaudio.load(flac_path)
    if signal.shape[0] > 1:
        signal = signal.mean(dim=0, keepdim=True)
    with torch.no_grad():
        emb = classifier.encode_batch(signal).squeeze().cpu().numpy()
    X.append(emb)
    rel = flac_path.relative_to(DATA_ROOT)
    utterances.append(str(rel))
    speakers.append(rel.parts[0])

X = np.vstack(X)
print("X shape:", X.shape)

  0%|          | 0/2620 [00:00<?, ?it/s]/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
100%|██████████| 2620/2620 [02:24<00:00, 18.09it/s]

X shape: (2620, 192)


In [5]:
np.save(OUT_DIR / "xvectors.npy",   X)
np.save(OUT_DIR / "utterances.npy", np.array(utterances))
np.save(OUT_DIR / "speakers.npy",   np.array(speakers))
print("Saved ECAPA embeddings to", OUT_DIR)

Saved ECAPA embeddings to ../outputs/librispeech_ecapa
